# Data Transformation — Star Schema for Power BI

This notebook transforms the raw DataCo supply chain dataset into a dimensional model optimized for analytics. The output is a star schema consisting of one fact table and four dimension tables, designed for direct consumption by Power BI. The local file structure mimics a typical medallion style architecture of Microsoft Fabric.

**Pipeline stage:** Cleaning and dimensional modeling  
**Input:** `../data/raw/DataCoSupplyChain.csv`  
**Output:** Five CSV files in `../data/cleaned/` (one fact table, four dimension tables)

## Why a Star Schema

A star schema separates measurable business events (the fact table) from descriptive context (the dimension tables). This structure reduces redundancy, improves query performance in Power BI, and makes DAX measures simpler to write. It is the standard pattern for enterprise reporting and is the recommended data model. 

## 1. Load Raw Data

The raw CSV produced by the ingestion notebook is loaded into a pandas DataFrame. Because of the small scale of the Kaggle dataset, pandas is appropriate. In a larger business context, Pyspark would be used along with SQL tables to handle a greater scale of computations. An output directory is created for the cleaned files if it does not already exist.

In [12]:
import pandas as pd
import os

os.makedirs("../data/cleaned", exist_ok=True)

raw_path = os.path.join("..", "data", "raw", "DataCoSupplyChain.csv")
df = pd.read_csv(raw_path, encoding="latin-1")


## 2. Clean and Enrich the Data

Three operations are performed here:

1. **Drop rows missing critical identifiers.** Records without an `Order Id`, `Customer Id`, or `Order Item Quantity` cannot be meaningfully analyzed and are removed.
2. **Convert date columns from string format to datetime.** This enables time-based filtering and date hierarchies in Power BI.
3. **Engineer the `Is Late` flag.** A binary column comparing actual vs. scheduled shipping days, which feeds the on-time delivery KPI in the final dashboard.

In [13]:
#Drop rows with null values in these columns
df.dropna(subset=["Order Id", "Customer Id", "Order Item Quantity"], inplace=True)

#Convert the columns from str to date-time format.
df["order date (DateOrders)"] = pd.to_datetime(df["order date (DateOrders)"])
df["shipping date (DateOrders)"] = pd.to_datetime(df["shipping date (DateOrders)"])

#Calculate new useful columns 
df["Is Late"] = (df["Days for shipping (real)"] > df["Days for shipment (scheduled)"]).astype(int)

### Verify Cleaning Results

After transformation, the data types and null counts are re-checked to confirm the cleaning logic worked as expected. A preview of the new `Is Late` column verifies the calculation.

In [14]:
# 5. Verify
print(df.dtypes)
print(df.isnull().sum())
df[["Is Late"]].head(10)

Unnamed: 0                                int64
Type                                        str
Days for shipping (real)                  int64
Days for shipment (scheduled)             int64
Benefit per order                       float64
Sales per customer                      float64
Delivery Status                             str
Late_delivery_risk                        int64
Category Id                               int64
Category Name                               str
Customer City                               str
Customer Country                            str
Customer Email                              str
Customer Fname                              str
Customer Id                               int64
Customer Lname                              str
Customer Password                           str
Customer Segment                            str
Customer State                              str
Customer Street                             str
Customer Zipcode                        

,Is Late
0,0
1,1
2,0
3,0
4,0
5,1
6,1
7,1
8,1
9,1


## 3. Build the Star Schema

The cleaned dataset is now split into one fact table and four dimension tables. Each dimension table is deduplicated so that descriptive attributes (customer names, product categories, regions) are stored only once and referenced from the fact table by ID.

### Fact Table — `fact_orders`

The fact table holds the measurable events of the business — in this case, individual order line items. It contains the foreign keys linking to each dimension table, plus the numeric measures (quantity, price, sales, profit ratio) that the Power BI dashboard will aggregate.

In [15]:
# Fact table
fact_cols = [
    "Order Id", "Customer Id", "Product Card Id",
    "order date (DateOrders)", "Order Item Quantity", "Order Item Product Price",
    "Sales", "Is Late", "Order Status", "Shipping Mode", "Days for shipping (real)",
    "Order Item Profit Ratio", "Order Country"
]
fact_orders = df[fact_cols]
fact_orders.to_csv("../data/cleaned/fact_orders.csv", index=False)
print(f"fact_orders: {fact_orders.shape}")

fact_orders: (180519, 13)


### Dimension Table — `dim_customer`

Customer attributes are extracted and deduplicated by `Customer Id`. This dimension supports customer segmentation analysis in the dashboard, such as revenue by segment or order volume by customer.

In [16]:
# Dim customer
dim_customer = df[["Customer Id", "Customer Fname", "Customer Lname",
                    "Customer Email", "Customer Segment"]].drop_duplicates()
dim_customer.to_csv("../data/cleaned/dim_customer.csv", index=False)
print(f"dim_customer: {dim_customer.shape}")

dim_customer: (20652, 5)


### Dimension Table — `dim_product`

Product attributes are extracted and deduplicated by `Product Card Id`. This dimension enables product-level analysis, including top-performing products and revenue by category or department.

In [17]:
# Dim product
dim_product = df[["Product Card Id", "Product Name", "Category Name",
                   "Department Name"]].drop_duplicates()
dim_product.to_csv("../data/cleaned/dim_product.csv", index=False)
print(f"dim_product: {dim_product.shape}")

dim_product: (118, 4)


### Dimension Table — `dim_region`

Geographic attributes are deduplicated at the country level, since each country belongs to exactly one region and market. This dimension powers the geographic visuals in the dashboard, including the map view and regional performance breakdowns.

In [18]:
# Dim region
dim_region = df[["Order Region", "Order Country", "Market"]].drop_duplicates(subset=["Order Country"])
dim_region.to_csv("../data/cleaned/dim_region.csv", index=False)
print(dim_region["Order Region"].value_counts())
print(f"dim_region: {dim_region.shape}")

Order Region
West Asia          16
West Africa        16
East Africa        14
South America      13
Southern Europe    11
Eastern Europe     11
Southeast Asia      9
South Asia          8
Central Africa      8
Northern Europe     8
Central America     8
Caribbean           8
Western Europe      7
Eastern Asia        6
North Africa        6
Southern Africa     5
Central Asia        5
Oceania             3
West of USA         1
Canada              1
Name: count, dtype: int64
dim_region: (164, 3)


### Dimension Table — `dim_date`

A dedicated date dimension is a standard pattern in star schemas. It enables time intelligence functions in Power BI such as year-over-year comparisons, quarter-to-date calculations, and consistent date hierarchies across all visuals.

In [27]:
# Dim date
dim_date = pd.DataFrame({"Date": df["order date (DateOrders)"].drop_duplicates()})
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Month"] = dim_date["Date"].dt.month
dim_date["Month Name"] = dim_date["Date"].dt.strftime("%B")
dim_date["Quarter"] = dim_date["Date"].dt.quarter
dim_date["Week"] = dim_date["Date"].dt.isocalendar().week
dim_date.to_csv("../data/cleaned/dim_date.csv", index=False)
print(f"dim_date: {dim_date.shape}")
print(dim_date["Date"].duplicated().sum())


dim_date: (65752, 6)
0


## 4. Exploratory Checks

Order status distribution is reviewed to confirm fraud detection candidates and validate the overall composition of the dataset. The distribution of suspected fraud across regions provides early input for the operational risk page of the dashboard.

In [39]:
print(df["Order Status"].value_counts())
print(df[df["Order Status"] == "SUSPECTED_FRAUD"].groupby("Order Region").size().describe())


Order Status
COMPLETE           59491
PENDING_PAYMENT    39832
PROCESSING         21902
PENDING            20227
CLOSED             19616
ON_HOLD             9804
SUSPECTED_FRAUD     4062
CANCELED            3692
PAYMENT_REVIEW      1893
Name: count, dtype: int64
count     23.000000
mean     176.608696
std      177.600609
min        6.000000
25%       68.500000
50%      147.000000
75%      206.500000
max      705.000000
dtype: float64


## 5. Confirm Outputs

Final check to verify all five CSV files were written to `../data/cleaned/` and are ready to be loaded into Microsoft Fabric or connected directly to Power BI.

In [26]:
# Verify all files exported
print(os.listdir("../data/cleaned"))

['dim_customer.csv', 'dim_date.csv', 'dim_product.csv', 'dim_region.csv', 'fact_orders.csv']


## Next Steps

The five cleaned CSVs form the foundation of the Power BI semantic model. From here:

1. Upload the files to a Microsoft Fabric Lakehouse, or connect Power BI Desktop to them directly.
2. Define relationships in Power BI between the fact table and each dimension table on their respective key columns.
3. Build DAX measures for the dashboard KPIs (on-time delivery rate, total revenue, late order count, inventory turnover).
4. Develop the multi-page dashboard covering executive summary, supplier performance, inventory analysis, and operational risk.